# CASE Personaplex Server Setup
## Google Colab Notebook for AI Voice Processing

This notebook sets up the Nvidia Personaplex server for the CASE self-balancing robot.

**Requirements:**
- GPU runtime (Runtime → Change runtime type → T4 GPU)
- Hugging Face account with Personaplex license accepted

## Step 1: Install Dependencies

In [ ]:
# Install Personaplex (Moshi)
!git clone https://github.com/kyutai-labs/moshi.git
!cd moshi && pip install -e .

# Install other dependencies
!pip install websockets pyngrok numpy soundfile librosa aiofiles

## Step 2: Authenticate with Hugging Face

In [ ]:
from huggingface_hub import notebook_login

# Login to Hugging Face
notebook_login()

## Step 3: Upload Server Code

In [ ]:
# Upload personaplex_server.py from your local machine
# Or paste the code directly:

server_code = '''
# Paste contents of personaplex_server.py here
'''

with open('personaplex_server.py', 'w') as f:
    f.write(server_code)

print("Server code saved!")

## Step 4: Start ngrok Tunnel

In [ ]:
from pyngrok import ngrok

# Optional: Set ngrok auth token for longer sessions
# ngrok.set_auth_token("YOUR_NGROK_TOKEN")

# Start ngrok tunnel on port 8765
public_url = ngrok.connect(8765, "http")
print(f"\n🌐 Public URL: {public_url}")
print(f"\n⚠️  Copy this URL and update ESP32 code:")
print(f"   const char* COLAB_SERVER_URL = \"{public_url.replace('http', 'ws')}/audio\";")
print("\n")

## Step 5: Start Personaplex Server

In [ ]:
# Start the WebSocket server
!python personaplex_server.py

: 

## Monitoring

Once the server is running:
1. The cell above will show connection logs
2. Each ESP32 connection will be logged
3. Audio processing will be shown in real-time

**Keep this notebook running** - if Colab disconnects, the server will stop.

**Session timeout:** Free Colab sessions timeout after ~90 minutes of inactivity.
Consider using Colab Pro for longer sessions.

## Testing

Test the WebSocket connection from your computer:

In [8]:
import asyncio
import websockets
import numpy as np

async def test_connection():
    # Replace with your ngrok URL
    uri = "http://172.28.0.12:8998"
    
    async with websockets.connect(uri) as websocket:
        # Send test audio (1 second of random noise)
        test_audio = (np.random.randn(16000) * 1000).astype(np.int16)
        await websocket.send(test_audio.tobytes())
        
        # Wait for response
        response = await websocket.recv()
        print(f"Received {len(response)} bytes")

# Uncomment to test
await test_connection()

InvalidURI: http://172.28.0.12:8998 isn't a valid URI: scheme isn't ws or wss